# Blok 3 · Zadanie grupowe: pobijcie swój model z Zajęć 2 🌳💳

**Sytuacja:** ten sam bank, te same dane (*Give Me Some Credit*), ta sama próbka 30 000 klientów i te same foldy co na Zajęciach 2. Na Zajęciach 2 doszliście do regresji logistycznej z PR-AUC ok. 0.35 — po sporej pracy z kodami 96/98, outlierami i brakami.

**Dziś pytanie brzmi:** czy modele drzewiaste dadzą bankowi **lepszy ranking ryzyka** i **równie wiarygodne PD** — i czy cała praca z danymi z Zajęć 2 jest jeszcze potrzebna?

## Zasady

* Grupy 2–3 osobowe, macie **ok. 45 minut**.
* Kroki 0–6. Komórki z `# TODO` uzupełniacie sami.
* **Każda decyzja → wiersz w dzienniku eksperymentów** (PR-AUC ± std, ROC-AUC, Brier, PR-AUC na treningu).
* Metryki jak na Zajęciach 2: **główna — PR-AUC** (ranking ryzyka przy rzadkiej klasie), **pomocnicza — Brier** (jakość samego PD).
* Narzędzia znacie z dzisiejszych notebooków `01_demo_drzewo_decyzyjne` i `02_demo_zespoly_drzew`.
* Jeśli coś liczy się dłużej niż minutę — zmniejszcie `n_estimators` / `n_iter`.

## Krok 0: Dane, CV, dziennik (~3 min)

Komórki są gotowe — uruchomcie je. Próbka i foldy są **identyczne** jak na Zajęciach 2, więc wyniki można porównywać wprost.

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import set_config
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
set_config(transform_output="pandas")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

# w Colabie:  !pip install catboost  (tylko jeśli chcecie spróbować CatBoosta w rozszerzeniach)
import xgboost as xgb

TARGET = "SeriousDlqin2yrs"

def wczytaj_kredyty():
    try:
        from scipy.io import arff
        df = pd.DataFrame(arff.loadarff("../02-ocena-i-dane/data/give-me-some-credit")[0])
    except FileNotFoundError:
        from sklearn.datasets import fetch_openml
        df = fetch_openml(data_id=45577, as_frame=True).frame
    df[TARGET] = df[TARGET].map(lambda v: v.decode() if isinstance(v, bytes) else v).astype(int)
    return df

df_pelny = wczytaj_kredyty()
df, _ = train_test_split(df_pelny, train_size=30_000, stratify=df_pelny[TARGET], random_state=RANDOM_STATE)
df = df.reset_index(drop=True)
X = df.drop(columns=TARGET)
y = df[TARGET]
print("próbka:", df.shape, f"  defaultów: {y.mean():.2%}")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
dziennik = []

def zapisz(nazwa, model, uwagi=""):
    t0 = time.time()
    r = cross_validate(model, X, y, cv=cv, scoring=["average_precision", "roc_auc", "neg_brier_score"],
                       return_train_score=True, n_jobs=-1)
    dziennik.append({"wariant": nazwa,
                     "PR-AUC": r["test_average_precision"].mean(),
                     "PR-AUC std": r["test_average_precision"].std(),
                     "ROC-AUC": r["test_roc_auc"].mean(),
                     "Brier": -r["test_neg_brier_score"].mean(),
                     "PR-AUC trening": r["train_average_precision"].mean(),
                     "czas [s]": round(time.time() - t0, 1),
                     "uwagi": uwagi})
    return pd.DataFrame(dziennik).set_index("wariant").round(4)

def pokaz_dziennik():
    return pd.DataFrame(dziennik).set_index("wariant").sort_values("PR-AUC", ascending=False).round(4)

Narzędzia z Zajęć 2 (zamiana kodów 96/98 na braki, winsoryzacja) i **punkt odniesienia**: najlepsza regresja logistyczna z rozwiązań Zajęć 2.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin

OPOZNIENIA = ["NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTime60-89DaysPastDueNotWorse", "NumberOfTimes90DaysLate"]

def kody_na_braki(X):
    X = X.copy()
    for c in OPOZNIENIA:
        X.loc[X[c] >= 96, c] = np.nan
    return X

class Winsoryzacja(BaseEstimator, TransformerMixin):
    def __init__(self, q=0.01):
        self.q = q
    def fit(self, X, y=None):
        A = np.asarray(X, dtype=float)
        self.dolne_ = np.nanquantile(A, self.q, axis=0)
        self.gorne_ = np.nanquantile(A, 1 - self.q, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.dolne_, self.gorne_)
    def set_output(self, *, transform=None):
        return self

def przygotowanie_z_zajec_2():
    return [FunctionTransformer(kody_na_braki), SimpleImputer(strategy="median", add_indicator=True), Winsoryzacja(0.01)]

from sklearn.dummy import DummyClassifier
zapisz("Dummy", DummyClassifier(strategy="prior"), "punkt odniesienia")
zapisz("LR: najlepsza z Zajęć 2", make_pipeline(*przygotowanie_z_zajec_2(), StandardScaler(), LogisticRegression(max_iter=3000)),
       "kody + mediana/wskaźnik + winsoryzacja + skalowanie")

## Krok 1: Drzewo decyzyjne (~6 min)

a. Dopiszcie do dziennika drzewo na **surowych** danych (bez imputera — drzewa przyjmują `NaN`), np. `max_depth=5`.

b. Nastrójcie drzewo `GridSearchCV` (`scoring="average_precision"`, ten sam `cv`): siatka `max_depth` × `min_samples_leaf`. Przy rzadkiej klasie spróbujcie **dużych** liści (50–300). Najlepszy wariant → dziennik.

💬 **Do dyskusji:** czy nastrojone drzewo bije regresję logistyczną z Zajęć 2? Spójrzcie też na kolumnę „PR-AUC trening”.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# TODO 1a: drzewo na surowych danych
# zapisz("drzewo: max_depth=5, surowe dane", ..., "")

# TODO 1b: GridSearchCV
# gs = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE), {...}, cv=cv, scoring="average_precision", n_jobs=-1).fit(X, y)
# print(gs.best_params_, gs.best_score_)
# zapisz(f"drzewo nastrojone {gs.best_params_}", gs.best_estimator_, "")

## Krok 2: Random Forest + kalibracja (~8 min)

a. Random Forest na surowych danych. Wypróbujcie co najmniej dwie wartości `min_samples_leaf` (np. 1 i 20). `n_estimators` 200–300 wystarczy.

b. Dla najlepszego lasu narysujcie **krzywą kalibracji** (`CalibrationDisplay.from_predictions` na predykcjach z `cross_val_predict`) razem z LR z Zajęć 2 i porównajcie **średnie przewidywane p** z odsetkiem defaultów.

💬 **Do dyskusji:** dlaczego `min_samples_leaf` ma tu tak duże znaczenie (pomyślcie o 6.7% defaultów i liściach z 1–2 klientami)? Czy las jest gorzej skalibrowany niż LR?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV

# TODO 2a: dwa lasy -> dziennik

# TODO 2b: krzywe kalibracji: najlepszy las vs LR z Zajęć 2
# fig, ax = plt.subplots(figsize=(7, 7))
# for nazwa, model in {...}.items():
#     p = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]
#     print(nazwa, "średnie p =", p.mean().round(4), " odsetek defaultów =", y.mean().round(4))
#     CalibrationDisplay.from_predictions(y, p, n_bins=10, strategy="quantile", ax=ax, name=nazwa)

## Krok 3: XGBoost — czy praca z Zajęć 2 jest jeszcze potrzebna? (~7 min)

a. XGBoost na **ustawieniach domyślnych** (pamiętajcie `n_jobs=1` w modelu — foldy liczą się równolegle).

b. XGBoost z mniejszym krokiem i płytszymi drzewami, np. `n_estimators=300, learning_rate=0.05, max_depth=4` — na **surowych** danych.

c. Ten sam model z (b), ale z przygotowaniem z Zajęć 2 przed nim: `make_pipeline(*przygotowanie_z_zajec_2(), xgb.XGBClassifier(...))`.

💬 **Do dyskusji:** (1) dlaczego domyślny XGBoost przegrywa? (2) czy winsoryzacja i imputacja coś dają modelowi drzewiastemu — i dlaczego?

In [ ]:
# TODO 3a: XGBoost domyślny

# TODO 3b: XGBoost lr=0.05, max_depth=4, 300 drzew — surowe dane

# TODO 3c: ten sam XGBoost + przygotowanie z Zajęć 2

## Krok 4: `RandomizedSearchCV` (~8 min)

Nastrójcie XGBoost losowym przeszukiwaniem: `n_iter` ok. 15, `scoring="average_precision"`, ten sam `cv`. Rozkłady (szablon poniżej) możecie zmienić.

💬 **Do dyskusji:** o ile poprawił się wynik względem kroku 3b? Czy ta poprawa jest większa niż odchylenie między foldami? Dlaczego `best_score_` jest **optymistyczny**?

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint, uniform

rozklady = {
    "n_estimators":     randint(100, 800),
    "learning_rate":    loguniform(0.01, 0.3),
    "max_depth":        randint(2, 8),
    "min_child_weight": loguniform(1, 50),
    "subsample":        uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.5, 0.5),
    "reg_lambda":       loguniform(0.1, 50),
}

# TODO 4: RandomizedSearchCV -> best_params_, best_score_, wiersz w dzienniku
# rs = RandomizedSearchCV(xgb.XGBClassifier(n_jobs=1, random_state=RANDOM_STATE), rozklady, n_iter=15, ...)

## Krok 5: Ograniczenia monotoniczne (~6 min)

Dział ryzyka pyta: „czy Wasz model może podnieść PD klientowi tylko dlatego, że zarabia więcej?”.

a. Wybierzcie **3–4 cechy**, dla których kierunek wpływu na ryzyko jest oczywisty, i w komórce markdown uzasadnijcie każdy kierunek **jednym zdaniem**.

b. Wytrenujcie XGBoost z kroku 3b (albo nastrojony z kroku 4) z `monotone_constraints={...}` — **poprzedzony zamianą kodów 96/98 na `NaN`**. Dopiszcie do dziennika.

c. (Szybki test) dla jednego klienta narysujcie PD w funkcji `MonthlyIncome` dla modelu bez i z ograniczeniami.

💬 **Do dyskusji:** czy bank powinien wybrać model z ograniczeniami, nawet jeśli ma o kilka tysięcznych niższe PR-AUC?

✍️ **Nasze ograniczenia i uzasadnienie:**

* `...`: +1 / −1, bo …
* `...`: +1 / −1, bo …
* `...`: +1 / −1, bo …

In [ ]:
# TODO 5b: XGBoost z ograniczeniami (po zamianie kodów na NaN)
# MONOTONICZNOSC = {"...": 1, "...": -1}
# model_mono = make_pipeline(FunctionTransformer(kody_na_braki), xgb.XGBClassifier(..., monotone_constraints=MONOTONICZNOSC))
# zapisz("XGBoost: kody -> NaN + ograniczenia monotoniczne", model_mono, "")

# TODO 5c: jeden klient, PD vs MonthlyIncome — bez i z ograniczeniami
# dochody = np.linspace(500, 20_000, 80)
# klient = X[X["MonthlyIncome"].notna()].iloc[[0]]
# Z = pd.concat([klient] * len(dochody), ignore_index=True); Z["MonthlyIncome"] = dochody
# plt.plot(dochody, model.predict_proba(Z)[:, 1], label=...)

## Krok 6: Finał — wybór, zapis i wnioski (~6 min)

1. Wyświetlcie dziennik posortowany po PR-AUC.
2. Wybierzcie **jeden** model dla banku. Jeśli używacie lasu bez kalibracji — sprawdźcie Brier; w razie potrzeby opakujcie model w `CalibratedClassifierCV`.
3. Wytrenujcie go na **całej próbce**, sprawdźcie, że przyjmuje **surowe dane**, i zapiszcie jako `grupa_XX.joblib` (XX = numer grupy).
4. Wnioski — 3 zdania.

In [ ]:
# TODO 6.1: dziennik
# pokaz_dziennik()

In [ ]:
import joblib

# TODO 6.3: model finalny
# model_finalny = ...
# model_finalny.fit(X, y)
# print(model_finalny.predict_proba(df_pelny.drop(columns=TARGET).sample(5, random_state=0))[:, 1].round(3))
# joblib.dump(model_finalny, "grupa_XX.joblib")

✍️ **Wasze wnioski (3 zdania):**

*(a) w porównaniu z Zajęciami 2 najwięcej dało…*

*(b) nie pomogło / okazało się niepotrzebne…*

*(c) bankowi polecamy…, bo… (ranking, kalibracja, zgodność z wiedzą domenową)*

## (Opcjonalnie) Rozszerzenia dla szybkich grup

* **LightGBM:** `lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, n_jobs=1, verbose=-1)` — czy wynik i czas różnią się od XGBoosta?
* **Ważność cech:** narysujcie `feature_importances_` (MDI) z lasu. Które cechy dominują? Dopiszcie kolumnę losowego szumu — na którym miejscu wyląduje?
* **Kalibracja:** czy `CalibratedClassifierCV(..., method="isotonic")` poprawia Brier najlepszego lasu?